In [ ]:
!pip install ccxt==4.3.90
!pip -q install ccxt pandas numpy


In [ ]:
import ccxt
import pandas as pd

exchange = ccxt.kraken()   # 예시: 다른 거래소
symbol = 'BTC/USDT'
timeframe = '1h'
limit = 500

ohlcv = exchange.fetch_ohlcv(symbol, timeframe=timeframe, limit=limit)

df = pd.DataFrame(
    ohlcv,
    columns=['timestamp', 'open', 'high', 'low', 'close', 'volume']
)
df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')

print(df.head())

            timestamp     open     high      low    close    volume
0 2026-02-13 04:00:00  66561.3  66650.0  66320.4  66360.6  2.083323
1 2026-02-13 05:00:00  66363.1  66476.6  66209.7  66209.7  2.718588
2 2026-02-13 06:00:00  66218.2  66363.0  66013.0  66363.0  0.680548
3 2026-02-13 07:00:00  66372.3  66575.2  66079.0  66217.5  2.027664
4 2026-02-13 08:00:00  66249.8  66715.3  66194.3  66661.9  2.255661


In [ ]:
import ccxt
import pandas as pd
import numpy as np
from datetime import datetime, timezone

exchange = ccxt.kraken({
    "enableRateLimit": True,   # ccxt 내부 레이트리밋 준수
})

# 심볼 로드 (Kraken은 심볼 표기가 거래소마다 달라서 로드해두는 게 안전)
markets = exchange.load_markets()
print("market count:", len(markets))

# Kraken에서 자주 쓰는 표기 예시 (계정/지역/시점에 따라 일부는 없을 수 있음)
candidates = ["BTC/USDT", "ETH/USDT", "SOL/USDT", "XBT/USDT", "BTC/USD", "ETH/USD", "SOL/USD"]

available = [s for s in candidates if s in markets]
print("available candidates:", available)

# 실제로 사용 가능한 심볼이 뭔지 확실히 보고 싶으면:
# print(list(markets.keys())[:200])



market count: 1479
available candidates: ['BTC/USDT', 'ETH/USDT', 'SOL/USDT', 'BTC/USD', 'ETH/USD', 'SOL/USD']


In [18]:
import ccxt
import pandas as pd
from datetime import datetime
import time
import warnings
warnings.filterwarnings('ignore')

# =====================
# 테스트할 거래소 목록 (한국 IP 차단 제외 후 후보군)
# =====================
EXCHANGES_TO_TEST = [
    'kraken', 'kucoin', 'gateio', 'mexc', 'bitfinex',
    'bitstamp', 'coinbase', 'huobi', 'okx', 'bybit',
    'phemex', 'bitget', 'bingx', 'woofipro', 'cryptocom'
]

SYMBOL    = 'BTC/USDT'
TIMEFRAME = '1d'
TEST_SINCE = '2015-01-01T00:00:00Z'  # 이 날짜부터 데이터 있는지 확인

results = []

print(f"{'Exchange':<15} {'Status':<12} {'First Date':<14} {'Days':<8} {'Spot Tickers'}")
print("-" * 65)

for ex_id in EXCHANGES_TO_TEST:
    try:
        exchange = getattr(ccxt, ex_id)({'enableRateLimit': True})

        # OHLCV 지원 여부 확인
        if not exchange.has.get('fetchOHLCV'):
            print(f"{ex_id:<15} {'No OHLCV':<12}")
            continue

        # 가장 오래된 데이터 요청
        since_ms = exchange.parse8601(TEST_SINCE)
        batch = exchange.fetch_ohlcv(SYMBOL, timeframe=TIMEFRAME, since=since_ms, limit=5)

        if not batch:
            print(f"{ex_id:<15} {'No data':<12}")
            continue

        first_date = pd.to_datetime(batch[0][0], unit='ms').date()

        # 현재까지 총 일수 계산
        today = datetime.utcnow().date()
        total_days = (today - first_date).days

        # 스팟 티커 수 확인
        try:
            markets = exchange.load_markets()
            spot_count = sum(1 for m in markets.values()
                             if m.get('spot') and m.get('quote') == 'USDT' and m.get('active'))
        except:
            spot_count = '?'

        results.append({
            'exchange': ex_id,
            'first_date': first_date,
            'total_days': total_days,
            'spot_usdt_tickers': spot_count
        })

        print(f"{ex_id:<15} {'✅ OK':<12} {str(first_date):<14} {total_days:<8} {spot_count}")

    except Exception as e:
        err = str(e)[:30]
        print(f"{ex_id:<15} {'❌ ' + err:<12}")

    time.sleep(0.5)  # rate limit 방지

# =====================
# 결과 요약
# =====================
if results:
    df_result = pd.DataFrame(results).sort_values('total_days', ascending=False)
    print("\n" + "=" * 65)
    print("🏆 Ranking by historical data length:")
    print("=" * 65)
    for i, row in df_result.iterrows():
        print(f"  {row['exchange']:<15} | from {row['first_date']} | {row['total_days']} days | {row['spot_usdt_tickers']} USDT tickers")

    best = df_result.iloc[0]
    print(f"\n📌 Best exchange: {best['exchange']} (from {best['first_date']}, {best['total_days']} days)")

Exchange        Status       First Date     Days     Spot Tickers
-----------------------------------------------------------------
kraken          ✅ OK         2024-03-16     720      46
kucoin          No data     
gateio          ✅ OK         2015-01-01     4082     2257
mexc            No data     
bitfinex        ✅ OK         2019-03-11     2552     76
bitstamp        No data     
coinbase        No data     
huobi           No data     
okx             No data     
bybit           ❌ bybit GET https://api.bybit.co
phemex          ✅ OK         2026-03-01     5        612
bitget          No data     
bingx           ❌ bingx {"code":100204,"msg":"da
woofipro        ❌ woofipro does not have market 
cryptocom       No data     

🏆 Ranking by historical data length:
  gateio          | from 2015-01-01 | 4082 days | 2257 USDT tickers
  bitfinex        | from 2019-03-11 | 2552 days | 76 USDT tickers
  kraken          | from 2024-03-16 | 720 days | 46 USDT tickers
  phemex          | from 